# Section 2: Traditional Non-Parallel Solution

This notebook implements the traditional baseline for the Chicago crime dataset using a non-parallel Python workflow. The goal is to clean and transform the data, recode detailed crime types into broader categories, and compute grouped crime counts by community area and crime category. The results from this notebook provide the baseline that will later be compared against the Spark RDD optimisation in Section 3.

## System specs

In [1]:
import platform
import psutil
import os

print("=== System Specs ===")
print(f"OS: {platform.system()} {platform.release()}")
print(f"CPU: {platform.processor()}")
print(f"CPU Cores (logical): {os.cpu_count()}")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.1f} GB")
print(f"Available RAM: {ram.available / (1024**3):.1f} GB")

=== System Specs ===
OS: Windows 11
CPU: ARMv8 (64-bit) Family 8 Model 1 Revision 201, Qualcomm Technologies Inc
CPU Cores (logical): 10
Total RAM: 15.6 GB
Available RAM: 4.3 GB


## Dataset and configuration

The traditional solution uses a subset of the Chicago crime dataset `Crimes_2020_2025`. This dataset is used so that the baseline can be implemented and measured using a non-parallel approach. We use a smaller subset as a prototype.

Working with the subset dataset makes it easier to observe where the performance bottlenecks occur, particularly during grouped aggregation

In [3]:
import os

import pandas as pd
import time
from pathlib import Path

start = time.time() # Start the timer for whole process

# Load in file

file_path = "Crimes_2020_2025.csv"

print("File exists:", os.path.exists(file_path))

File exists: True


## Step 1: Load the subset data

The first step is to load the dataset into a dataframe. This gives us a single baseline representation of the data that can be cleaned and analysed.

We record the loading times and initial memory usage so we keep track of the cost and be able to compare later.

In [6]:
# Start the timing for the file loading step
start_load = time.time()

# Read only the columns we need for the analysis
# Parsed the date column straight after the datetime format
df = pd.read_csv(
    file_path,
    usecols=[
        "Date",
        "Primary Type",
        "Description",
        "Location Description",
        "Arrest",
        "District",
        "Ward",
        "Community Area",
        "Domestic"
    ],
    parse_dates=["Date"],
    date_format="%m/%d/%Y %I:%M:%S %p"
)

# This stops the timing for the file loading step
load_time = time.time() - start_load

print("Loaded dataset shape:", df.shape)

# This is the raw memory usage of the dataframe in MB 
raw_memory = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Raw dataframe memory usage: {raw_memory:.2f} MB")
print(f"Load time: {load_time:.2f} seconds")

df.head()

Loaded dataset shape: (1420664, 9)
Raw dataframe memory usage: 299.01 MB
Load time: 18.65 seconds


,Date,Primary Type,Description,Location Description,Arrest,Domestic,District,Ward,Community Area
0,2022-07-29 03:39:00,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,True,False,10.0,25.0,30.0
1,2023-01-03 16:44:00,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,True,False,11.0,28.0,26.0
2,2020-08-10 09:45:00,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,14.0,1.0,24.0
3,2023-09-06 17:00:00,CRIMINAL DAMAGE,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,1.0,42.0,32.0
4,2023-09-06 11:00:00,THEFT,OVER $500,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,1.0,4.0,32.0


## Step 2: Clean and transform the data

Here we clean the rows with missing critical values and the date column is changed and can be used later for analysis. We extracted the hour of the incident the day, month, and year. We converted the selected location fields into numerical form where possible. This produces a cleaner and more structured dataset that can be used later for analysis

In [7]:
start_clean = time.time() # Start timing

# Drops rows with any missing fields needed for the analysis.
df = df.dropna(subset=["Date", "Primary Type", "Community Area"]).copy()

# Extracts time based features from the Date column for analysis.
df["incident_hour"] = df["Date"].dt.hour
df["incident_day"] = df["Date"].dt.day_name()
df["incident_month"] = df["Date"].dt.month
df["incident_year"] = df["Date"].dt.year

# Converts selected columns to numeric values
# Implemented error handling instead of crashing
df["District"] = pd.to_numeric(df["District"], errors="coerce").astype("Int64")
df["Ward"] = pd.to_numeric(df["Ward"], errors="coerce").astype("Int64")
df["Community Area"] = pd.to_numeric(df["Community Area"], errors="coerce")


# Converts these into boolean values
df["Arrest"] = df["Arrest"].astype(str).str.strip().str.upper().eq("TRUE")
df["Domestic"] = df["Domestic"].astype(str).str.strip().str.upper().eq("TRUE")

# Drops rows if community area is missing
df = df.dropna(subset=["Community Area"]).copy()

# Converts community area into an int
df["Community Area"] = df["Community Area"].astype(int)

# Stop timing
clean_time = time.time() - start_clean

clean_memory = df.memory_usage(deep=True).sum() / (1024 ** 2)

# Prints the summary
print("Numbers of rows and columns after and transformation:" , df.shape)

print(f"Cleaned dataframe memory usage: {clean_memory:.2f} MB")
print(f"Clean and transform time: {clean_time:.2f} seconds")

df.head()

Numbers of rows and columns after and transformation: (1420565, 13)
Cleaned dataframe memory usage: 404.85 MB
Clean and transform time: 6.92 seconds


,Date,Primary Type,Description,Location Description,Arrest,Domestic,District,Ward,Community Area,incident_hour,incident_day,incident_month,incident_year
0,2022-07-29 03:39:00,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,True,False,10,25,30,3,Friday,7,2022
1,2023-01-03 16:44:00,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,True,False,11,28,26,16,Tuesday,1,2023
2,2020-08-10 09:45:00,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,14,1,24,9,Monday,8,2020
3,2023-09-06 17:00:00,CRIMINAL DAMAGE,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,1,42,32,17,Wednesday,9,2023
4,2023-09-06 11:00:00,THEFT,OVER $500,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,1,4,32,11,Wednesday,9,2023


## Step 3: Recode detailed crime types into broader crime categories

The raw `Primary Type` field contains many detailed labels, this makes a high level analysis harder to interpret.   
We simplified this analysis by creating broader categories for the crimes such as property related, violent crimes, public order and more. This allows for easier analysis and comparisons.

In [8]:
# All the different crime types
def recode_crime(primary_type):
    if primary_type in [
        "HOMICIDE", "ASSAULT", "BATTERY", "ROBBERY",
        "KIDNAPPING", "INTIMIDATION", "STALKING"
    ]:
        return "VIOLENT CRIME"
    elif primary_type in [
        "CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
        "SEX OFFENSE", "HUMAN TRAFFICKING"
    ]:
        return "SEXUAL / TRAFFICKING"
    elif primary_type in [
        "THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
        "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
        "DECEPTIVE PRACTICE", "ARSON"
    ]:
        return "PROPERTY CRIME"
    elif primary_type in ["NARCOTICS", "OTHER NARCOTIC VIOLATION"]:
        return "DRUG OFFENSE"
    elif primary_type in [
        "PROSTITUTION", "GAMBLING", "LIQUOR LAW VIOLATION",
        "PUBLIC INDECENCY", "OBSCENITY", "RITUALISM",
        "PUBLIC PEACE VIOLATION"
    ]:
        return "PUBLIC ORDER"
    elif primary_type in [
        "WEAPONS VIOLATION", "CONCEALED CARRY LICENSE VIOLATION"
    ]:
        return "WEAPONS OFFENSE"
    elif primary_type == "OFFENSE INVOLVING CHILDREN":
        return "CRIMES AGAINST CHILDREN"
    elif primary_type == "INTERFERENCE WITH PUBLIC OFFICER":
        return "LAW ENFORCEMENT RELATED"
    elif primary_type == "OTHER OFFENSE":
        return "MISCELLANEOUS CRIME"
    elif primary_type == "NON-CRIMINAL":
        return "NON-CRIMINAL"
    else:
        return "UNCLASSIFIED"

start_recode = time.time()


df["Crime Category"] = df["Primary Type"].apply(recode_crime)

recode_time = time.time() - start_recode

print("\nCrime category distribution:")

print(df["Crime Category"].value_counts().head(15))
print(f"Recode time: {recode_time:.2f} seconds")



Crime category distribution:
Crime Category
PROPERTY CRIME             762507
VIOLENT CRIME              441950
MISCELLANEOUS CRIME         90731
WEAPONS OFFENSE             49260
DRUG OFFENSE                36555
SEXUAL / TRAFFICKING        16690
CRIMES AGAINST CHILDREN     10889
PUBLIC ORDER                 8412
LAW ENFORCEMENT RELATED      3552
NON-CRIMINAL                   19
Name: count, dtype: int64
Recode time: 1.26 seconds


## Step 4: Main grouped aggregation

The main analysis in this baseline is to count the number of crimes in each group : `(Community Area, Crime Category)`.

This grouping of aggregation is the central baseline results because it shows how crime categories are distributed across different areas and provides clearest task for the spark RDD later.

In [9]:
start_agg = time.time()

main_counts = (
    df.groupby(["Community Area", "Crime Category"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

agg_time = time.time() - start_agg

result_memory = main_counts.memory_usage(deep=True).sum() / (1024 ** 2)

print("\nTop 20 grouped counts by Community Area and Crime Category:")
print(main_counts.head(20))
print(f"Aggregation time: {agg_time:.2f} seconds")
print(f"Grouped result memory usage: {result_memory:.2f} MB")


Top 20 grouped counts by Community Area and Crime Category:
    Community Area  Crime Category  count
0                8  PROPERTY CRIME  39972
1               28  PROPERTY CRIME  34511
2               25  PROPERTY CRIME  32360
3               32  PROPERTY CRIME  30608
4               24  PROPERTY CRIME  28470
5               25   VIOLENT CRIME  27526
6                6  PROPERTY CRIME  24549
7               43  PROPERTY CRIME  23706
8               43   VIOLENT CRIME  18375
9               71  PROPERTY CRIME  18226
10              22  PROPERTY CRIME  17803
11              44  PROPERTY CRIME  17263
12               7  PROPERTY CRIME  16765
13              69  PROPERTY CRIME  16663
14              49  PROPERTY CRIME  16362
15              23  PROPERTY CRIME  15447
16              29   VIOLENT CRIME  15387
17              29  PROPERTY CRIME  15131
18              28   VIOLENT CRIME  14524
19               3  PROPERTY CRIME  14113
Aggregation time: 0.33 seconds
Grouped result memory usag

## Step 5: Additional supporting analysis

We created more outputs to support our analysis. They provide more insight into our dataset and is not used for the benchmarking.

### Hourly crime distribution

This output shows how crime categories are distributed across different hours of the day. Property crime is appears in all times.

In [10]:
hourly_counts = (
    df.groupby(["Crime Category", "incident_hour"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

print("\nTop 10 hourly crime counts:")
print(hourly_counts.head(10))


Top 10 hourly crime counts:
   Crime Category  incident_hour  count
0  PROPERTY CRIME              0  59431
1  PROPERTY CRIME             12  48788
2  PROPERTY CRIME             17  41756
3  PROPERTY CRIME             18  40694
4  PROPERTY CRIME             15  40554
5  PROPERTY CRIME             16  39768
6  PROPERTY CRIME             14  37931
7  PROPERTY CRIME             19  37154
8  PROPERTY CRIME              9  35583
9  PROPERTY CRIME             13  35422


### Arrests rate by crime category

This calculates the proportion of records in each crime category that resulted in an arrest. It shows us an additional perspective on how different categories relate to arrest rates outcomes.

In [11]:
arrest_rate = (
    df.groupby("Crime Category")["Arrest"]
      .mean()
      .reset_index(name="arrest_rate")
      .sort_values("arrest_rate", ascending=False)
      .reset_index(drop=True)
)

print("\nArrest rate by crime category:")
print(arrest_rate.head(10))


Arrest rate by crime category:
            Crime Category  arrest_rate
0             DRUG OFFENSE     0.974504
1  LAW ENFORCEMENT RELATED     0.893300
2          WEAPONS OFFENSE     0.640926
3             PUBLIC ORDER     0.609011
4             NON-CRIMINAL     0.368421
5      MISCELLANEOUS CRIME     0.159901
6            VIOLENT CRIME     0.137443
7  CRIMES AGAINST CHILDREN     0.098172
8     SEXUAL / TRAFFICKING     0.066087
9           PROPERTY CRIME     0.054615


### Domestic rate by crime category

This calculates the proportion of records in each crime category that are marked as domestic incidents. This provides supporting view of how the nature of incidents varies across categories.

In [12]:
domestic_rate = (
    df.groupby("Crime Category")["Domestic"]
      .mean()
      .reset_index(name="domestic_rate")
      .sort_values("domestic_rate", ascending=False)
      .reset_index(drop=True)
)

print("\nDomestic rate by crime category:")
print(domestic_rate.head(10))


Domestic rate by crime category:
            Crime Category  domestic_rate
0  CRIMES AGAINST CHILDREN       0.793094
1            VIOLENT CRIME       0.411424
2      MISCELLANEOUS CRIME       0.387398
3     SEXUAL / TRAFFICKING       0.232415
4           PROPERTY CRIME       0.063742
5             NON-CRIMINAL       0.052632
6             PUBLIC ORDER       0.037328
7  LAW ENFORCEMENT RELATED       0.018300
8          WEAPONS OFFENSE       0.006537
9             DRUG OFFENSE       0.002106


## Timing summary

We created separate timers to record for loading, cleaning and transformation, recoding and grouped aggregation. 

Breaking the runtime into stages makes it easier to identify the most expensive part of the traditional workflow and provides the evidence needed to justify the Section 3 optimisation.

In [18]:
summary = pd.DataFrame({
    "Stage": [
        "Load dataset",
        "Clean and transform",
        "Recode crime categories",
        "Grouped aggregation"
    ],
    "Time (seconds)": [
        load_time,
        clean_time,
        recode_time,
        agg_time
    ]
})

total_time = load_time + clean_time + recode_time + agg_time

print(f"Total baseline time: {total_time:.2f} seconds")
summary

Total baseline time: 27.16 seconds


,Stage,Time (seconds)
0,Load dataset,18.652217
1,Clean and transform,6.921025
2,Recode crime categories,1.264209
3,Grouped aggregation,0.326568


## Memory summary

The Memory usage is reported for the traditional baseline so that the space requirements of the non-parallel workflow are visible alongside the timing results. We can compare the timing and memory figures to provide the baseline performance information required for comparison with Spark.

In [14]:
print(f"Raw dataframe memory: {raw_memory:.2f} MB")
print(f"Cleaned dataframe memory: {clean_memory:.2f} MB")
print(f"Grouped result memory: {result_memory:.2f} MB")

Raw dataframe memory: 299.01 MB
Cleaned dataframe memory: 404.85 MB
Grouped result memory: 0.06 MB


## Bottleneck identification

The grouped aggregation step was chosen as the main optimisation target, even though it is not the slowest step in the baseline timings. In the traditional solution, most of the time is spent loading and cleaning the data, which are largely sequential tasks and involve file I/O and parsing.

However, aggregation is still the most suitable step to optimise because it involves grouping records and combining values, which becomes increasingly expensive as the dataset grows. 

This type of operation maps directly to the MapReduce model, where data can be processed in parallel by key.

In contrast, loading and cleaning are less suitable for this type of optimisation. As a result, the aggregation step was selected as the bottleneck to optimise in Section 3, as it provides the clearest opportunity to demonstrate the benefits of Spark’s distributed processing.

## Bottleneck identification

The timing results show that loading (18.65s) dominates total runtime, followed by clean/transform (6.92s), recode (1.26s), and grouped aggregation (0.33s)

Loading is inherently sequential — it is bound by disk read speed and cannot be parallelised by MapReduce. It is therefore not a suitable optimisation target.

The **clean/transform (6.92s)** and **recode (1.26s)** steps are the most suitable MapReduce targets because:
- Each row is processed **independently** — no row depends on any other, making this a perfect Map operation
- Work can be freely split across partitions with no coordination needed
- These steps scale linearly with data volume, meaning they become increasingly dominant as the dataset grows beyond the subset

The **grouped aggregation (0.33s)** is also a natural MapReduce target — it maps directly to `reduceByKey`, where records sharing the same `(Community Area, Crime Category)` key are counted in parallel across partitions.

These two steps — **clean/recode (Map)** and **grouped aggregation (Reduce)** — are therefore selected as the optimisation targets for Section 3.
